Phase I EDA

In [1]:
# 1. Imports
import pandas as pd
import joblib
import json
from datetime import datetime
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import json
from datetime import datetime
import numpy as np
os.makedirs('assets', exist_ok=True)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# 2. Load dataset
df = pd.read_csv(r'C:\Users\ABCD\_ML projects(SDS)\SDS-CP036-powercast\advanced\submissions\team-members\lakshay-yadav\assets\Tetuan City power consumption.csv')

In [3]:
print(df.head())

In [4]:
print(df.info())

In [5]:
print(df.describe())

In [6]:
print(df.isnull().sum())

In [7]:
# Check and report rows with invalid DateTime before dropping
if 'DateTime' in df.columns:
    invalid_dt = df[pd.to_datetime(df['DateTime'], errors='coerce').isna()]
    print(f'Number of rows with invalid DateTime: {len(invalid_dt)}')
    if len(invalid_dt) > 0:
        print('Sample of rows with invalid DateTime:')
        print(invalid_dt.head())
else:
    print('DateTime column not found. Please check the column name.')

In [8]:
# Parse and validate DateTime column
if 'DateTime' in df.columns:
    df['DateTime'] = pd.to_datetime(df['DateTime'], errors='coerce')
    # No invalid DateTime rows detected, so no need to drop
    df.sort_values('DateTime', inplace=True)
    print('DateTime column parsed and DataFrame sorted.')
    print(df[['DateTime']].head())
else:
    print('DateTime column not found. Please check the column name.')

In [9]:
# Check frequency and consistency of DateTime
if 'DateTime' in df.columns:
    # Calculate time differences between consecutive timestamps
    df['time_diff'] = df['DateTime'].diff().dt.total_seconds() / 60  # in minutes
    print('Time difference between consecutive rows (in minutes):')
    print(df['time_diff'].value_counts())
    # Visualize gaps in time series
    plt.figure(figsize=(10,4))
    plt.plot(df['DateTime'], df['time_diff'], marker='o', linestyle='-', alpha=0.5)
    plt.title('Time Difference Between Consecutive Timestamps')
    plt.ylabel('Minutes')
    plt.xlabel('DateTime')
    plt.show()
    # Check for missing intervals
    missing_intervals = df[df['time_diff'] != 10]
    print(f'Number of missing or irregular intervals: {len(missing_intervals)}')
    if len(missing_intervals) > 0:
        print('Sample of irregular intervals:')
        print(missing_intervals[['DateTime', 'time_diff']].head())
else:
    print('DateTime column not found. Please check the column name.')

In [10]:
# Inspect the irregular interval and suggest handling
if 'time_diff' in df.columns:
    irregular = df[df['time_diff'] != 10]
    print('Details of the irregular interval:')
    print(irregular)
    # Suggestion: For a single gap, you can either drop, impute, or leave as is
    print('If this is a minor gap, you can drop the row, fill with interpolation, or simply note it in your report.')
else:
    print('time_diff column not found. Please run the previous cell to compute time differences.')

### Analysis of Irregular Interval

- The apparent irregular interval occurs only at the first row where `time_diff` is NaN; this is expected because there is no previous timestamp for comparison.
- No actual gaps are present; the series follows a consistent 10‑minute cadence throughout.
- No imputation or removal is required for this check.

In [11]:
# Data Integrity Check: Duplicates and Uniqueness
print("Duplicate DateTime:", df['DateTime'].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())
if df['DateTime'].duplicated().sum() == 0 and df.duplicated().sum() == 0:
    print("No duplicates detected.")
else:
    print("Duplicates detected. Consider dropping or investigating.")


### Findings from Data Integrity Check

- No duplicate `DateTime` values or full-row duplicates were detected.
- Temporal uniqueness is preserved across the dataset.
- No deduplication is required before further analysis.


In [12]:
# Visualize trends in power consumption across zones
plt.figure(figsize=(15,6))
plt.plot(df['DateTime'], df['Zone 1 Power Consumption'], label='Zone 1', alpha=0.7)
plt.plot(df['DateTime'], df['Zone 2  Power Consumption'], label='Zone 2', alpha=0.7)
plt.plot(df['DateTime'], df['Zone 3  Power Consumption'], label='Zone 3', alpha=0.7)
plt.xlabel('DateTime')
plt.ylabel('Power Consumption')
plt.title('Power Consumption Trends Across Zones')
plt.legend()
plt.tight_layout()
plt.savefig(r'assets/power_consumption_trends.png')  # Save plot to assets folder
plt.show()

In [13]:
# Seasonality Profiles: Hour-of-Day and Day-of-Week Averages
zones = ['Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']
df['hour'] = df['DateTime'].dt.hour
df['dow'] = df['DateTime'].dt.dayofweek

hourly_avg = df.groupby('hour')[zones].mean()
ax = hourly_avg.plot(figsize=(10,4), title='Average Power Consumption by Hour of Day')
plt.tight_layout(); plt.savefig('assets/avg_by_hour.png'); plt.show()

dow_avg = df.groupby('dow')[zones].mean()
ax = dow_avg.plot(figsize=(10,4), title='Average Power Consumption by Day of Week (0=Mon)')
plt.tight_layout(); plt.savefig('assets/avg_by_dow.png'); plt.show()

# Weekday × Hour Heatmap of Power Consumption (Zone 1 and all zones)
pivot = df.pivot_table(index=df['DateTime'].dt.dayofweek,
                       columns=df['DateTime'].dt.hour,
                       values='Zone 1 Power Consumption',
                       aggfunc='mean')
sns.heatmap(pivot, cmap='viridis')
plt.title('Mean Consumption by Weekday (0=Mon) and Hour — Zone 1')
plt.xlabel('Hour')
plt.ylabel('Weekday (0=Mon)')
plt.tight_layout(); plt.savefig('assets/weekday_hour_heatmap_zone1.png'); plt.show()

zones = ['Zone 1 Power Consumption','Zone 2  Power Consumption','Zone 3  Power Consumption']
fig, axes = plt.subplots(1, 3, figsize=(18,4), sharex=True, sharey=True)
for ax, z in zip(axes, zones):
    pivot = df.pivot_table(index=df['DateTime'].dt.dayofweek,
                           columns=df['DateTime'].dt.hour,
                           values=z,
                           aggfunc='mean')
    sns.heatmap(pivot, cmap='viridis', ax=ax, cbar=False)
    ax.set_title(z.replace('  ', ' '))
    ax.set_xlabel('Hour'); ax.set_ylabel('Weekday (0=Mon)')
plt.tight_layout(); plt.savefig('assets/weekday_hour_heatmap_all_zones.png'); plt.show()


### Findings from Seasonality Profiles

- Hour‑of‑day profiles show clear daily cycles with consistent peaks and troughs across zones.
- Day‑of‑week profiles indicate moderate differences, suggesting a weekday–weekend effect.
- Weekday × hour heatmaps provide a 2D view highlighting evening peaks and lower overnight usage; weekdays are generally higher than weekends.
- These seasonal patterns motivate including daily‑scale context in downstream modeling.


In [14]:
# Outlier Scan: Quick z-score counts per column
import numpy as np
cols_check = ['Zone 1 Power Consumption','Zone 2  Power Consumption','Zone 3  Power Consumption',
              'Temperature','Humidity','Wind Speed','general diffuse flows','diffuse flows']
z = (df[cols_check] - df[cols_check].mean())/df[cols_check].std()
outlier_counts = (z.abs() > 3).sum()
print("Num outliers (|z|>3) per column:\n", outlier_counts)

# Optional: cap extreme values for visualization stability (no permanent change)
# df_capped = df.copy()
# for c in cols_check:
#     cap = 3*df[c].std()
#     df_capped[c] = df[c].clip(df[c].mean()-cap, df[c].mean()+cap)


### Findings from Outlier Scan

- A z-score threshold of 3 flags a manageable number of extremes in both power and weather variables.
- The presence of outliers is consistent with real-world sensor behavior.
- No changes are applied at this stage; handling (e.g., capping or filtering) can be considered during preprocessing.

In [15]:
# Autocorrelation Analysis: ACF and PACF (hourly resampled)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

zones = ['Zone 1 Power Consumption','Zone 2  Power Consumption','Zone 3  Power Consumption']
for i, zone in enumerate(zones, start=1):
    series_hourly = df.set_index('DateTime')[zone].resample('H').mean().dropna()
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    plot_acf(series_hourly, lags=200, ax=ax[0]); ax[0].set_title(f'ACF ({zone}, hourly)')
    plot_pacf(series_hourly, lags=60, ax=ax[1]); ax[1].set_title(f'PACF ({zone}, hourly)')
    plt.tight_layout(); 
    plt.savefig(f'assets/{zone.replace(" ", "_").lower()}_acf_pacf.png'); 
    plt.show()

### Findings from Autocorrelation Analysis

- ACF exhibits strong daily periodicity on hourly‑resampled series (peaks near 24, 48, … hours).
- ACF decay suggests persistence in the load process, while PACF highlights impactful short lags.
- Evidence supports both short‑term dependence and daily seasonality across all zones.


In [19]:
# Optional: STL Decomposition for Seasonality (Zone 1, hourly)
from statsmodels.tsa.seasonal import STL
zone_hourly = df.set_index('DateTime')['Zone 1 Power Consumption'].resample('H').mean().dropna()
res = STL(zone_hourly, period=24).fit()
fig = res.plot(); 
fig.set_size_inches(10,7)
plt.tight_layout(); 
plt.savefig('assets/zone1_stl.png'); 
plt.show()


### Findings from STL Decomposition (Optional)

- Seasonal component displays a stable daily pattern, consistent with the autocorrelation results.
- Trend and seasonal components dominate residuals, indicating strong, repeatable structure.
- Decomposition supports including daily‑scale context in subsequent modeling.


In [20]:
# Visualize and analyze relationships between power consumption and weather features
weather_features = ['Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows']
power_zones = ['Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']

# Scatter plots for each weather feature vs each zone
for zone in power_zones:
    for feature in weather_features:
        plt.figure(figsize=(7,4))
        plt.scatter(df[feature], df[zone], alpha=0.3)
        plt.xlabel(feature)
        plt.ylabel(zone)
        plt.title(f'{zone} vs {feature}')
        plt.tight_layout()
        plt.savefig(f'assets/{zone.replace(" ", "_").lower()}_vs_{feature.replace(" ", "_").lower()}.png')
        plt.show()

# Correlation heatmap
corr = df[weather_features + power_zones].corr()
plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Between Weather Features and Power Consumption')
plt.tight_layout()
plt.savefig('assets/correlation_heatmap.png')
plt.show()

### Findings from Power Consumption and Weather Feature Analysis

**1. Temperature:**
- Temperature exhibits a strong positive association with power consumption across zones; higher temperatures align with higher demand, consistent with cooling loads.

**2. Humidity:**
- Humidity shows wide dispersion with no clear monotonic trend, indicating weak influence on demand.

**3. Wind Speed:**
- Wind speed displays weak relationships with consumption, with concentrations at low/high values and no consistent trend.

**4. Diffuse Flows & General Diffuse Flows:**
- Scatter patterns narrow at higher radiation values, suggesting lower or more stable consumption when solar radiation is higher (e.g., daylight or solar offset effects).

**5. Correlation Heatmap:**
- Correlations confirm temperature as the strongest positive driver; humidity and wind speed are weak to negative; diffuse radiation features are low to moderate.

**Overall Conclusion:**
- Across zones, temperature emerges as the most influential weather feature, while humidity, wind speed, and diffuse radiation play smaller roles. These observations prioritize temperature‑related features for subsequent modeling.

In [21]:
# Lag Effect Analysis: Create lagged features and check correlations
lag_steps = [1, 2, 3]  # Lag by 1, 2, and 3 intervals (10, 20, 30 minutes)
weather_features = ['Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows']
power_zones = ['Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']

# Create lagged columns for each feature
for lag in lag_steps:
    for col in weather_features + power_zones:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Collect lagged correlations in a DataFrame
import numpy as np
lag_corrs = []
for zone in power_zones:
    for feature in weather_features:
        for lag in lag_steps:
            corr = df[zone].corr(df[f'{feature}_lag{lag}'])
            lag_corrs.append({
                'Zone': zone,
                'Feature': feature,
                'Lag': lag,
                'Correlation': np.round(corr, 3)
            })
lag_corrs_df = pd.DataFrame(lag_corrs)

# Display the lagged correlation table
lag_corrs_df_pivot = lag_corrs_df.pivot_table(index=['Zone','Feature'], columns='Lag', values='Correlation')
display(lag_corrs_df_pivot)

# Example: Visualize lag effect for Zone 1 power vs lagged temperature
plt.figure(figsize=(7,4))
plt.scatter(df['Temperature_lag1'], df['Zone 1 Power Consumption'], alpha=0.3)
plt.xlabel('Temperature (lag 1)')
plt.ylabel('Zone 1 Power Consumption')
plt.title('Zone 1 Power Consumption vs Temperature (lag 1)')
plt.tight_layout()
plt.savefig('assets/zone1_power_vs_temperature_lag1.png')
plt.show()

# Weather→Power Lag Heatmap and Best-Lag Table
import numpy as np
weather_features = ['Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows']
zones = ['Zone 1 Power Consumption','Zone 2  Power Consumption','Zone 3  Power Consumption']
lags = list(range(1, 25))  # 1–24 steps (10-min each ≈ up to 4h)

# Build lagged correlation matrix: rows = feature@lag, cols = zones
rows = []
for feature in weather_features:
    for k in lags:
        row = { 'Feature@Lag': f'{feature}_lag{k}' }
        for z in zones:
            row[z] = df[feature].shift(k).corr(df[z])
        rows.append(row)
lagcorr_df = pd.DataFrame(rows).set_index('Feature@Lag')

# Heatmap
plt.figure(figsize=(10, 10))
sns.heatmap(lagcorr_df, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5, linewidths=0.3, linecolor='white')
plt.title('Lagged Correlation: Weather (lagged) → Power (current)')
plt.tight_layout(); plt.savefig('assets/weather_lag_to_power_heatmap.png'); plt.show()

# Best-lag table per feature→zone
best = []
for feature in weather_features:
    for z in zones:
        vals = [df[feature].shift(k).corr(df[z]) for k in lags]
        arr = np.array(vals)
        best_k = int(lags[int(np.nanargmax(np.abs(arr)))])
        best_corr = float(arr[best_k-1])
        best.append({ 'Feature': feature, 'Zone': z, 'Best Lag (steps)': best_k, 'Correlation': round(best_corr, 3) })

best_df = pd.DataFrame(best)
display(best_df.sort_values(['Zone', 'Feature']).reset_index(drop=True))

### Lag Effect Analysis Findings

- Temperature lagged features show the strongest association with current power usage, especially at short lags.
- Humidity, wind speed, and diffuse radiation lags exhibit weaker or inconsistent effects.
- Weather→power lag heatmap (1–24 steps) identifies the most relevant short delays by feature and zone; best‑lag table summarizes these.
- Similar lag patterns are observed across zones, supporting temperature as the most influential lagged driver for forecasting.

In [22]:
# Lookback Window and Forecast Horizon Proposal (based on ACF/PACF & seasonality)
import json

# Infer sampling in minutes (expected ~10)
sampling_minutes = int(df['DateTime'].diff().dt.total_seconds().dropna().median() // 60)
daily_period_steps = int(24 * 60 / sampling_minutes)

# Candidate lookbacks (short-term + full daily cycle)
LOOKBACK_CANDIDATES_STEPS = [
    int(6 * 60 / sampling_minutes),   # 6h
    int(12 * 60 / sampling_minutes),  # 12h
    daily_period_steps                # 24h
]

# Candidate forecast horizons (near-term to short-term)
FORECAST_HORIZON_STEPS = [
    int(60 / sampling_minutes),       # 1h ahead
    int(2 * 60 / sampling_minutes),   # 2h ahead
    int(6 * 60 / sampling_minutes)    # 6h ahead
]

summary = {
    "sampling_minutes": sampling_minutes,
    "daily_period_steps": daily_period_steps,
    "lookback_candidates_steps": LOOKBACK_CANDIDATES_STEPS,
    "forecast_horizon_steps": FORECAST_HORIZON_STEPS,
    "notes": [
        "Daily seasonality observed; include 24h lookback.",
        "PACF highlights short impactful lags; include 6–12h lookbacks.",
        "Horizons at 1–6h balance near-term accuracy and operational usefulness."
    ]
}

print("Sampling (minutes):", sampling_minutes)
print("Daily period (steps):", daily_period_steps)
print("Lookback candidates (steps):", LOOKBACK_CANDIDATES_STEPS)
print("Forecast horizons (steps):", FORECAST_HORIZON_STEPS)

# Save for Phase 2 handoff
with open('assets/phase1_eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Saved proposal to assets/phase1_eda_summary.json")


### Lookback and Forecast Horizon Decision

- Candidate lookback windows: 6h, 12h, and 24h, reflecting short-term effects and the daily cycle indicated by ACF/PACF.
- Candidate forecast horizons: 1h, 2h, and 6h ahead to balance near-term accuracy and operational relevance.
- These selections align with the detected daily seasonality and short-term autocorrelation, and will guide sequence construction in Phase 2.


# Phase 2

📌Week 2 | Sequence Generation

In [25]:
# -----------------------------
# ✅ Feature Engineering Prep (before sequence generation)
# -----------------------------
# Copy dataset to new feature dataframe
df_feat = df.copy()

# Extract cyclical time features
df_feat["hour"] = df_feat["DateTime"].dt.hour
df_feat["weekday"] = df_feat["DateTime"].dt.weekday

df_feat["hour_sin"] = np.sin(2 * np.pi * df_feat["hour"] / 24)
df_feat["hour_cos"] = np.cos(2 * np.pi * df_feat["hour"] / 24)
df_feat["weekday_sin"] = np.sin(2 * np.pi * df_feat["weekday"] / 7)
df_feat["weekday_cos"] = np.cos(2 * np.pi * df_feat["weekday"] / 7)

# Keep only numeric features (inputs + targets)
numeric_df = df_feat.select_dtypes(include=[np.number])

# Keep datetime column separately
datetime_col = df_feat["DateTime"].copy()

print("Numeric DF shape:", numeric_df.shape)
print("Datetime col length:", len(datetime_col))

In [26]:
# 🚨 CRITICAL FIX: DataLoader factory for consistency
def create_dataloader_factory(batch_size=64, shuffle_train=True):
    """
    Factory function for consistent DataLoader creation across all experiments
    """
    def create_loaders(X, y, shuffle=False):
        dataset = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y))
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)
    return create_loaders
# Enhanced DataLoader creation with factory
dl_factory = create_dataloader_factory(batch_size=batch_size, shuffle_train=True)
train_loader = dl_factory(X_train_t, y_train_t, shuffle=True)
val_loader = dl_factory(X_val_t, y_val_t, shuffle=False)
test_loader = dl_factory(X_test_t, y_test_t, shuffle=False)


In [27]:
# 🚨 CRITICAL FIX: Comprehensive sample count logging
def log_dataset_configuration(lookback, horizon, feature_set, splits, save_dir):
    """
    Log all dataset configuration details for reproducibility
    """
    config_log = {
        'timestamp': str(pd.Timestamp.now()),
        'lookback_window': lookback,
        'prediction_horizon': horizon,
        'feature_set': feature_set,
        'splits': {
            'train': {
                'samples': len(splits['train']), 
                'shape': splits['train'].shape,
                'memory_mb': splits['train'].nbytes / (1024 * 1024)
            },
            'val': {
                'samples': len(splits['val']), 
                'shape': splits['val'].shape,
                'memory_mb': splits['val'].nbytes / (1024 * 1024)
            },
            'test': {
                'samples': len(splits['test']), 
                'shape': splits['test'].shape,
                'memory_mb': splits['test'].nbytes / (1024 * 1024)
            }
        },
        'total_sequences': sum(len(s) for s in splits.values()),
        'total_memory_mb': sum(s.nbytes for s in splits.values()) / (1024 * 1024)
    }
    # Save to file
    config_file = os.path.join(save_dir, f"config_log_lb{lookback}_hr{horizon}.json")
    with open(config_file, 'w') as f:
        json.dump(config_log, f, indent=2)
    print(f"📊 Configuration logged: {config_log['total_sequences']} total sequences")
    print(f"   Memory usage: {config_log['total_memory_mb']:.2f} MB")
    print(f"   Config saved: {config_file}")
    return config_log
# Log configuration for current setup
splits_dict = {
    'train': X_train_t,
    'val': X_val_t, 
    'test': X_test_t
}
config_log = log_dataset_configuration(
    lookback=lb, 
    horizon=hr, 
    feature_set=f"allzones_{X_train_t.shape[-1]}features",
    splits=splits_dict,
    save_dir=save_dir
)


In [28]:
# -----------------------------
# ✅ Load Phase 1 summary (lookbacks & horizons)
# -----------------------------
assets_dir = "assets"
summary_path = os.path.join(assets_dir, "phase1_eda_summary.json")

with open(summary_path, "r") as f:
    eda_summary = json.load(f)

lookback_options = eda_summary.get("lookbacks", [144])   # fallback: 24h (10-min data)
horizon_options  = eda_summary.get("horizons", [6])      # fallback: 1h ahead

print("Available lookbacks:", lookback_options)
print("Available horizons:", horizon_options)


# -----------------------------
# ✅ Multi-zone Sequence Maker
# -----------------------------
def make_sequences_multizone(data: pd.DataFrame, target_cols: list, lookback: int, horizon: int):
    """
    Generate sequences for multiple target columns (zones).
    
    Args:
        data (pd.DataFrame): Input dataframe with numeric features + targets.
        target_cols (list): List of target column names.
        lookback (int): Number of past timesteps to use as features.
        horizon (int): Number of timesteps ahead to forecast.
    
    Returns:
        X (np.ndarray): Input features (n_samples, lookback, n_features).
        y (np.ndarray): Multi-zone targets (n_samples, horizon, n_targets).
    """
    values = data.values
    target_indices = [data.columns.get_loc(c) for c in target_cols]

    X, y = [], []
    for i in range(len(values) - lookback - horizon + 1):
        X.append(values[i : i + lookback])
        y.append(values[i + lookback : i + lookback + horizon, target_indices])

    return np.array(X), np.array(y)


# -----------------------------
# ✅ DateTime Sequence Maker
# -----------------------------
def make_datetime_sequences(datetime_series, lookback, horizon):
    """
    Generate sequences of DateTime values corresponding to input sequences.
    """
    dt_sequences = []
    n_samples = len(datetime_series) - lookback - horizon + 1
    for i in range(n_samples):
        dt_seq = datetime_series[i:i+lookback].values
        dt_sequences.append(dt_seq)
    return np.array(dt_sequences)


# -----------------------------
# ✅ Generate & Save Sequences (All Zones Together)
# -----------------------------
target_columns = [
    "Zone 1 Power Consumption",
    "Zone 2  Power Consumption",
    "Zone 3  Power Consumption"
]

save_dir = os.path.join(assets_dir, "prepared_sequences")
os.makedirs(save_dir, exist_ok=True)

for lookback in lookback_options:
    for horizon in horizon_options:
        # Generate numeric sequences
        X, y = make_sequences_multizone(
            numeric_df,
            target_cols=target_columns,
            lookback=lookback,
            horizon=horizon
        )

        # Generate corresponding DateTime sequences
        dt_seq = make_datetime_sequences(datetime_col, lookback, horizon)

        # Save sequences
        np.save(os.path.join(save_dir, f"X_allzones_lb{lookback}_hr{horizon}.npy"), X)
        np.save(os.path.join(save_dir, f"y_allzones_lb{lookback}_hr{horizon}.npy"), y)
        np.save(os.path.join(save_dir, f"datetime_lb{lookback}_hr{horizon}.npy"), dt_seq)

        print(
            f"✅ Saved sequences for lb{lookback}, hr{horizon} | "
            f"Shapes -> X: {X.shape}, y: {y.shape}, DateTime: {dt_seq.shape}"
        )

### Sequence Generation  

- Sliding window approach used to create supervised learning sequences from time series.  
- Candidate lookback and horizon values applied across all zones.  
- Generated sequences saved as `.npy` files for reuse in future phases.  

In [29]:
# -----------------------------
# ✅ Normalization (StandardScaler for all zones together)
# -----------------------------
from sklearn.preprocessing import StandardScaler
import joblib

scaler_dir = os.path.join(assets_dir, "scalers")
os.makedirs(scaler_dir, exist_ok=True)

def normalize_and_save(X: np.ndarray, lookback: int, horizon: int):
    n_samples, lb, n_features = X.shape
    X_reshaped = X.reshape(-1, n_features)   # flatten for scaler

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_reshaped).reshape(n_samples, lb, n_features)

    # Save scaler
    scaler_filename = f"scaler_X_allzones_lb{lookback}_hr{horizon}.pkl"
    scaler_path = os.path.join(scaler_dir, scaler_filename)
    joblib.dump(scaler, scaler_path)

    print(f"✅ Normalized (all zones) | Scaler saved at: {scaler_path}")
    return X_scaled


# -----------------------------
# ✅ Apply normalization on saved sequences
# -----------------------------
seq_dir = os.path.join(assets_dir, "prepared_sequences")

for lb in lookback_options:
    for hr in horizon_options:
        X_path = os.path.join(seq_dir, f"X_allzones_lb{lb}_hr{hr}.npy")
        y_path = os.path.join(seq_dir, f"y_allzones_lb{lb}_hr{hr}.npy")
        
        X = np.load(X_path)
        y = np.load(y_path)

        # Normalize X
        X_scaled = normalize_and_save(X, lb, hr)

        # Overwrite normalized X
        np.save(X_path, X_scaled)
        print(f"✅ Overwritten with normalized data: {X_path}")

### Normalization of Sequences  

- Inputs (X) normalized using StandardScaler to stabilize training across zones.  
- Targets (y) kept raw for interpretability of predictions.  
- Zone-specific scalers saved for later inverse transformation during evaluation and deployment.  

### Train/Validation/Test Split & Tensor Conversion  

- Sequences (`X`, `y`) split into **70% train, 15% validation, 15% test**.  
- Converted splits into **PyTorch tensors** for model input.  
- Also saved NumPy arrays for reproducibility.

In [30]:
# %%
# -----------------------------
# ✅ Step 3: Train/Val/Test Split + Tensor Conversion
# -----------------------------
import torch

# Ratios
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15

seq_dir = os.path.join(assets_dir, "prepared_sequences")
split_dir = os.path.join(assets_dir, "splits")
os.makedirs(split_dir, exist_ok=True)

# Dictionary to hold splits
splits = {}

for lb in lookback_options:
    for hr in horizon_options:
        # Load sequences
        X = np.load(os.path.join(seq_dir, f"X_allzones_lb{lb}_hr{hr}.npy"))
        y = np.load(os.path.join(seq_dir, f"y_allzones_lb{lb}_hr{hr}.npy"))

        n_samples = len(X)
        train_end = int(n_samples * train_ratio)
        val_end = train_end + int(n_samples * val_ratio)

        # �� CRITICAL FIX: Save split indices for reproducibility
        split_indices = {
            'lookback': lb,
            'horizon': hr,
            'train_end': train_end,
            'val_end': val_end,
            'train_samples': train_end,
            'val_samples': val_end - train_end,
            'test_samples': n_samples - val_end,
            'total_samples': n_samples,
            'split_ratios': {'train': train_ratio, 'val': val_ratio, 'test': test_ratio}
        }
        
        # Save split indices to disk
        split_indices_file = os.path.join(split_dir, f"split_indices_lb{lb}_hr{hr}.json")
        with open(split_indices_file, 'w') as f:
            json.dump(split_indices, f, indent=2)
        
        print(f"💾 Split indices saved: {split_indices_file}")

        # Split
        X_train, X_val, X_test = X[:train_end], X[train_end:val_end], X[val_end:]
        y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]

        # Convert to tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        X_val_t   = torch.tensor(X_val, dtype=torch.float32)
        y_val_t   = torch.tensor(y_val, dtype=torch.float32)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32)
        y_test_t  = torch.tensor(y_test, dtype=torch.float32)

        # Save NumPy splits
        key = f"lb{lb}_hr{hr}"
        np.save(os.path.join(split_dir, f"X_train_{key}.npy"), X_train)
        np.save(os.path.join(split_dir, f"y_train_{key}.npy"), y_train)
        np.save(os.path.join(split_dir, f"X_val_{key}.npy"), X_val)
        np.save(os.path.join(split_dir, f"y_val_{key}.npy"), y_val)
        np.save(os.path.join(split_dir, f"X_test_{key}.npy"), X_test)
        np.save(os.path.join(split_dir, f"y_test_{key}.npy"), y_test)

        # Store in dictionary
        splits[key] = {
            "train": (X_train_t, y_train_t),
            "val":   (X_val_t, y_val_t),
            "test":  (X_test_t, y_test_t),
        }

        print(f"✅ Split done for {key} | Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

### DataLoader Preparation  

- Wrapped train/validation/test splits into **TensorDataset** objects.  
- Created **PyTorch DataLoaders** with batching (default: 64).  
- Enables efficient mini-batch training and validation.  

In [31]:
# %%
# -----------------------------
# ✅ Step 4: DataLoader Preparation
# -----------------------------
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64
dataloaders = {}

for key, sets in splits.items():
    X_train_t, y_train_t = sets["train"]
    X_val_t, y_val_t     = sets["val"]
    X_test_t, y_test_t   = sets["test"]

    # TensorDatasets
    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds   = TensorDataset(X_val_t, y_val_t)
    test_ds  = TensorDataset(X_test_t, y_test_t)

    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    dataloaders[key] = {
        "train": train_loader,
        "val": val_loader,
        "test": test_loader,
    }

    print(f"✅ DataLoaders ready for {key} | Batches -> "
          f"Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")
